<a href="https://colab.research.google.com/github/eeeewyz/agent/blob/main/9_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Lab: Agentic Workflows

In this lab, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [ ]:
# =========================
# Imports
# =========================

# --- Standard library
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [ ]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [ ]:
CLIENT = Client()

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:o4-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

它不是负责“做研究”的 Agent，而是负责“规划研究流程”的 Agent。
所以整个结构其实是：

用户给 Topic
   ↓
planner_agent
   ↓
LLM 分析任务
   ↓
生成 research plan
   ↓
[
  step 1,
  step 2,
  step 3,
  ...
]

然后后面的系统才会按照这个 plan，去调用真正的：

Research Agent
Writer Agent
Editor Agent

4. 为什么要求 atomic + executable
Each step should be atomic, executable

这是非常典型的 Agent Planning 要求。

Atomic = 一步最好只干一件事情。

5. 下面这些 ✅ / 🚫 是在“防止模型跑偏”

例如：

DO NOT include irrelevant tasks like
"create CSV", "set up a repo", "install packages"

意思是：

这是一个 research workflow，不要突然变成 software engineering workflow。

LLM 很容易“过度规划”。

比如用户问：

Research recent RAG techniques

模型可能自动脑补：

1. Create project folder
2. Install dependencies
3. Create CSV
4. Create GitHub repo
5. Search papers

但前四步跟你的目标没关系。

所以这是一个 constraint。

然后：

DO include real research-related tasks
(e.g., search, summarize, draft, revise)

这是给模型 positive examples。

return ONLY the Python list

这一条非常重要，主要不是为了语言效果，而是为了程序稳定性。

因为后面：

steps_str = response.choices[0].message.content.strip()


steps = ast.literal_eval(steps_str)

假如模型返回：

Sure! Here is the research plan:


["Search papers", "Write summary"]

那么：

ast.literal_eval()

可能直接报错。

所以要求：

ONLY Python list

实际上是在建立一个简单的 output contract：

最后一条为什么指定 Markdown report
The final step should be to generate a Markdown document
containing the complete research report.

这是在给 Planner 一个明确的 final goal / termination condition。

也就是：

什么时候整个 workflow 算完成？

答案：

最后产生完整 Markdown research report。

In [ ]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """


    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create(
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1,
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [ ]:
# Test your code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

current_time： 给 Research Agent 当前日期，让它能正确理解“最新、最近、今年”等时间相关要求。对需要搜索 Web 或 arXiv 的任务尤其有用。

为什么用 user prompt： 这个作业把角色说明、工具说明和具体任务统一放进一个 user message 里。虽然里面写了 “You are a research agent”，但真正决定消息角色的是 {"role": "user"}。

In [ ]:
# GRADED FUNCTION: research_agent

def research_agent(
    task: str,
    model: str = "openai:gpt-4o",
    return_messages: bool = False
):
    """
    使用工具完成研究任务。

    参数:
        task (str): 需要完成的研究任务。
        model (str): 使用的语言模型。
        return_messages (bool): 是否同时返回传给模型的 messages。

    返回:
        str 或 tuple:
        默认返回模型最终生成的文本。
        如果 return_messages=True，则返回 (content, messages)。
    """

    print("==================================")
    print("🔍 Research Agent")
    print("==================================")

    # 获取当前日期
    current_time = datetime.now().strftime("%Y-%m-%d")

    ### START CODE HERE ###

    # 构造传给 Research Agent 的 user prompt
    prompt = f"""
    You are a research agent responsible for completing research tasks using the available tools.

    Available tools:
    - Use arxiv_tool to search for academic papers.
    - Use tavily_tool to perform general web searches.
    - Use wikipedia_tool to retrieve encyclopedic information.

    Use these tools when appropriate to gather reliable and relevant information.

    Research task:
    {task}

    Current date:
    {current_time}

    Provide a clear and concise research response based on the information you find.
    """

    # 构造传给 LLM 的 messages
    # 注意：根据这个作业的要求，这里使用的是 user 角色
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 定义 Research Agent 可以使用的工具
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool
    ]

    # 调用 LLM，并把工具一起传给模型
    response = CLIENT.chat.completions.create(
        # 指定使用的模型
        model=model,

        # 传入前面构造好的 user message
        messages=messages,

        # 把 Research Agent 可以使用的工具提供给模型
        tools=tools,

        # 让模型根据当前任务自动决定是否调用工具，以及调用哪个工具
        tool_choice="auto",

        # 最多允许 6 轮 LLM ↔ Tool 的交互
        max_turns=6
    )

    ### END CODE HERE ###

    # 从模型返回结果中提取最终文本
    content = response.choices[0].message.content

    print("✅ Output:\n", content)

    # 如果 return_messages=True，
    # 则除了最终结果之外，同时返回最开始传给模型的 messages
    return (content, messages) if return_messages else content

In [ ]:
# Test your code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 Here are three key references from the recent academic papers, along with a brief summary of each:

1. **Two-dimensional magnetic interactions in LaFeAsO**:
   - **Summary**: This paper provides evidence from inelastic neutron scattering measurements that the magnetic interactions in LaFeAsO are two-dimensional. It highlights the similarity of these interactions to those found in \textit{A}Fe$_2$As$_2$ based materials but notes a significantly smaller interlayer and intralayer exchange ratio. This finding suggests that the effective dimensionality of the magnetic system in parent compounds of iron arsenides is variable, which might influence superconductivity.

2. **Goldilocks mixing in oceanic shear-induced turbulent overturns**:
   - **Summary**: The authors introduce a new parameterization based on the ratio of Thorpe and Ozmidov scales for the turbulent flux coefficient in stratified flows. The study identifies three phases of shear-induced turbulence, w

## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [ ]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    执行写作任务，例如起草、扩展或总结文本。
    """

    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###

    # 创建 system prompt：
    # 定义模型的身份以及 Writer Agent 主要负责什么
    system_prompt = """
    You are a writing agent specialized in generating
    well-structured academic or technical content.

    Your responsibilities include drafting, expanding,
    and summarizing text.
    """

    # System message：规定 Writer Agent 的角色和职责
    system_msg = {
        "role": "system",
        "content": system_prompt
    }

    # User message：传入本次具体需要执行的写作任务
    user_msg = {
        "role": "user",
        "content": task
    }

    # 把 system message 和 user message 放进 messages
    messages = [system_msg, user_msg]

    ### END CODE HERE ###

    # 调用 LLM
    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1.0
    )

    # 返回模型生成的文本内容
    return response.choices[0].message.content

In [ ]:
# Test your code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [ ]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    执行编辑任务，例如反思、批评、修改和改进已有文本。
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")

    ### START CODE HERE ###

    # 创建 system prompt：
    # 定义模型为 Editor Agent，并说明它负责审阅、批评和改进已有草稿
    system_prompt = """
    You are an editor agent specialized in reviewing and improving existing drafts.

    Your responsibilities include:
    - Reflecting on the quality and structure of the draft.
    - Identifying unclear, inaccurate, or weak parts.
    - Providing constructive critique when necessary.
    - Revising the draft to improve clarity, coherence, accuracy, and readability.
    - Preserving the original meaning and intent whenever possible.
    """

    # System message：定义 Editor Agent 的角色和职责
    system_msg = {
        "role": "system",
        "content": system_prompt
    }

    # User message：传入本次具体需要编辑或修改的任务
    user_msg = {
        "role": "user",
        "content": task
    }

    # 将 system 和 user message 一起传给模型
    messages = [system_msg, user_msg]

    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7
    )

    # 返回 Editor Agent 最终生成的内容
    return response.choices[0].message.content

In [ ]:
# Test your code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

👉 **Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [ ]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

Planner 生成步骤
        ↓
Executor 逐个读取 step
        ↓
LLM 决定交给哪个 Agent
        ↓
把之前结果组成 context
        ↓
调用对应 Agent
        ↓
结果存入 history
        ↓
继续下一步

def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    # 先调用 planner_agent，根据 topic 生成一个研究计划
    # plan_steps 是一个字符串列表，每个元素代表一个需要执行的步骤
    plan_steps = planner_agent(topic)

    # 为了控制运行时间，最多只执行 4 个步骤
    max_steps = 4

    # 如果 limit_steps=True，就只保留前 4 个步骤
    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    # history 用来保存已经执行过的步骤
    # 每一项的结构是：
    # (原始 step, 执行这个 step 的 agent 名字, agent 的输出结果)
    history = []

    print("==================================")
    print("🎯 Executor Agent")
    print("==================================")

    # 逐个执行 planner 生成的每一个 step
    for i, step in enumerate(plan_steps):

        # 构造一个 prompt，让 LLM 判断：
        # 1. 当前这个 step 应该交给哪个 agent
        # 2. 这个 agent 实际应该执行什么 task
        #
        # 要求 LLM 只返回 JSON，方便后面程序解析
        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """

        # 调用 LLM，让它决定当前 step 应该由哪个 agent 执行
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": agent_decision_prompt
                }
            ],

            # temperature=0：
            # 让 agent 选择尽量稳定，不希望每次随机选择不同 agent
            temperature=0,
        )

        # 取出 LLM 返回的文本
        raw_content = response.choices[0].message.content

        # 有时 LLM 可能返回：
        # ```json
        # {...}
        # ```
        # clean_json_block 会把外面的 Markdown 代码块去掉
        cleaned_json = clean_json_block(raw_content)

        # 把 JSON 字符串转换成 Python 字典
        agent_info = json.loads(cleaned_json)

        # 从字典中取出：
        # 应该调用哪个 agent
        # 以及这个 agent 应该执行的 task
        agent_name = agent_info["agent"]
        task = agent_info["task"]

        # 把之前所有步骤的执行结果组合成 context
        #
        # 这样下一步 agent 不只是看到自己的 task，
        # 还能够知道前面的 agent 已经完成了什么
        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}"
            for j, (s, a, r) in enumerate(history)
        ])

        # 构造真正传给目标 agent 的任务
        #
        # enriched_task = 当前任务 + 前面步骤的结果
        # 这样可以让多个 agent 之间共享上下文
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(
            f"\n🛠️ Executing with agent: `{agent_name}` "
            f"on task: {task}"
        )

        # 检查 LLM 选择的 agent 是否真的存在于 agent_registry 中
        if agent_name in agent_registry:

            # agent_registry 是类似这样的映射：
            #
            # {
            #     "research_agent": research_agent,
            #     "writer_agent": writer_agent,
            #     "editor_agent": editor_agent
            # }
            #
            # 所以：
            # agent_registry["research_agent"]
            # 实际上拿到的就是 research_agent 函数

            # 根据 agent_name 动态调用对应的 agent
            # 并把 enriched_task 传进去
            output = agent_registry[agent_name](enriched_task)

            # 把这一步的执行记录保存到 history
            history.append(
                (step, agent_name, output)
            )

        else:
            # 如果 LLM 返回了不存在的 agent 名字，
            # 就生成一个错误提示，而不是直接让程序崩溃
            output = f"⚠️ Unknown agent: {agent_name}"

            # 同样把错误结果保存到 history
            history.append(
                (step, agent_name, output)
            )

        # 打印当前步骤的执行结果
        print(f"✅ Output:\n{output}")

    # 所有步骤执行结束之后，
    # 返回完整的执行历史
    return history

输入 topic

plan_steps = Planner(topic)

history = []

for step in plan_steps:

    # 1. 判断这一步该交给哪个 Agent
    agent_name, task = LLM_decide_agent(step)

    # 2. 读取前面步骤的结果
    context = build_context(history)

    # 3. 把“当前任务 + 历史结果”组合起来
    enriched_task = context + task

    # 4. 调用对应 Agent
    output = agent_registry[agent_name](enriched_task)

    # 5. 保存执行结果
    history.append(step, agent_name, output)

返回 history

In [ ]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]

    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}"
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [ ]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: Search Wikipedia for an overview of the ensemble Kalman filter
🔍 Research Agent
✅ Output:
 The Ensemble Kalman Filter (EnKF) is a recursive filter specifically designed for handling problems with a large number of variables, such as those encountered in discretizations of partial differential equations within geophysical models. It originated as an adaptation of the standard Kalman filter, tailored for large-scale problems by substituting the covariance matrix with the sample covariance. EnKF is essential in ensemble forecasting as a data assimilation tool.

This filter is akin to the particle filter, with an ensemble member functioning similarly to a particle. However, the EnKF operates under the assumption that all associated probability distributions are Gaussian, which allows it to be more efficient than the particle filter in applicable situations.

The EnKF functions as a Monte Carlo method for executing Bayesian 

## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>